# Brain Age Model Exploration

**Goal:** Build the best possible lean brain age model using only features from the 5 core tasks (`d1, d2, nb1, nb2, rest_closed`) and our standard biomarkers.

**Constraints:**
- Only core tasks (new participants won't have other tasks)
- Only biomarkers from our metrics pipeline: A0, Alpha, Beta, Theta, Delta, Gamma, ST4, T2, VC9, responsetime, accuracy
- Model must be lean (runs in real-time in OSP)
- Target: r > 0.7 on healthy population

**Key questions:**
1. Which features correlate best with age?
2. Why does the current model give negative gap for healthy?
3. Can we do better than r=0.61?

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict, KFold, LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Libraries loaded.')

## 1. Load Data & Build Feature Matrix

In [ ]:
df = pd.read_pickle('df_hrv.pkl')

CORE_TASKS = ['d1', 'd2', 'nb1', 'nb2', 'rest_closed']
BIO_COLS = ['A0', 'Alpha', 'Beta', 'Theta', 'Delta', 'Gamma', 'ST4', 'T2', 'VC9']
BEHAV_COLS = ['responsetime', 'accuracy']
ALL_FEAT_COLS = BIO_COLS + BEHAV_COLS

# Filter to core tasks
df_core = df[df['task_level'].isin(CORE_TASKS)].copy()

# Convert to numeric
for col in ALL_FEAT_COLS:
    df_core[col] = pd.to_numeric(df_core[col], errors='coerce')
df_core['age'] = pd.to_numeric(df_core['age'], errors='coerce')

print(f'Core task rows: {len(df_core)}')
print(f'Users: {df_core["username"].nunique()}')

In [ ]:
# Build per-user, per-task feature means
user_task_means = df_core.groupby(['username', 'task_level'])[ALL_FEAT_COLS].mean()

# Pivot to wide format: columns = feature_task
records = []
for username, group in user_task_means.groupby(level=0):
    row = {'username': username}
    for (_, task), vals in group.iterrows():
        for feat in ALL_FEAT_COLS:
            row[f'{feat}_{task}'] = vals[feat]
    records.append(row)

df_wide = pd.DataFrame(records)

# Add user-level info
user_info = df.groupby('username').agg({
    'age': 'first', 'mmse_group': 'first', 'mmse': 'first',
    'f_dopa_group': 'first', 'group': 'first'
}).reset_index()
user_info['age'] = pd.to_numeric(user_info['age'], errors='coerce')
user_info['mmse'] = pd.to_numeric(user_info['mmse'], errors='coerce')

df_wide = df_wide.merge(user_info, on='username', how='left')

# Also add cross-task differences (delta features)
for feat in ALL_FEAT_COLS:
    # nb2 - nb1 (difficulty increase)
    c1, c2 = f'{feat}_nb2', f'{feat}_nb1'
    if c1 in df_wide.columns and c2 in df_wide.columns:
        df_wide[f'{feat}_diff_nb2_nb1'] = df_wide[c1] - df_wide[c2]
    # d2 - d1
    c1, c2 = f'{feat}_d2', f'{feat}_d1'
    if c1 in df_wide.columns and c2 in df_wide.columns:
        df_wide[f'{feat}_diff_d2_d1'] = df_wide[c1] - df_wide[c2]
    # nb2 - rest_closed (cognitive vs rest)
    c1, c2 = f'{feat}_nb2', f'{feat}_rest_closed'
    if c1 in df_wide.columns and c2 in df_wide.columns:
        df_wide[f'{feat}_diff_nb2_rest'] = df_wide[c1] - df_wide[c2]
    # overall mean across all core tasks
    task_cols = [f'{feat}_{t}' for t in CORE_TASKS if f'{feat}_{t}' in df_wide.columns]
    if task_cols:
        df_wide[f'{feat}_mean_all'] = df_wide[task_cols].mean(axis=1)

print(f'Wide dataframe: {df_wide.shape}')
print(f'Users with age: {df_wide["age"].notna().sum()}')

In [ ]:
# Define population subsets
has_age = df_wide['age'].notna()
is_healthy = df_wide['mmse_group'] == 'healthy'
is_mci = df_wide['mmse_group'] == 'MCI'
is_md = df_wide['mmse_group'] == 'MD'
is_unclassified = df_wide['mmse_group'] == '0'
is_fdopa_pos = df_wide['f_dopa_group'].astype(str).str.contains('ositive', na=False)

# Exclude F-DOPA positive (Parkinson's) from training
not_parkinsons = ~is_fdopa_pos

# For training: healthy + unclassified (presumed healthy) — NO MCI/MD
train_mask = has_age & (is_healthy | is_unclassified) & not_parkinsons
# For healthy-only evaluation
healthy_mask = has_age & is_healthy & not_parkinsons

print(f'Training pool (healthy + unclassified, no PD): {train_mask.sum()}')
print(f'Healthy only (MMSE>=24): {healthy_mask.sum()}')
print(f'MCI: {(has_age & is_mci).sum()}')
print(f'MD: {(has_age & is_md).sum()}')
print(f'F-DOPA positive (excluded): {is_fdopa_pos.sum()}')
print(f'\nTraining age range: {df_wide.loc[train_mask, "age"].min():.0f} - {df_wide.loc[train_mask, "age"].max():.0f}')
print(f'Healthy age range: {df_wide.loc[healthy_mask, "age"].min():.0f} - {df_wide.loc[healthy_mask, "age"].max():.0f}')

## 2. Feature-Age Correlations (Healthy Population)

Let's find which features correlate most strongly with age in the healthy population.

In [ ]:
# Get all numeric feature columns (exclude metadata)
meta_cols = ['username', 'age', 'mmse_group', 'mmse', 'f_dopa_group', 'group']
feat_cols = [c for c in df_wide.columns if c not in meta_cols and df_wide[c].dtype in ['float64', 'int64']]

# Compute correlations with age in healthy population
df_h = df_wide[healthy_mask].copy()
correlations = []
for col in feat_cols:
    valid = df_h[[col, 'age']].dropna()
    if len(valid) >= 30:
        r, p = spearmanr(valid[col], valid['age'])
        rp, pp = pearsonr(valid[col], valid['age'])
        correlations.append({
            'feature': col, 
            'spearman_r': r, 'spearman_p': p,
            'pearson_r': rp, 'pearson_p': pp,
            'n': len(valid),
            'abs_r': abs(r)
        })

df_corr = pd.DataFrame(correlations).sort_values('abs_r', ascending=False)
print(f'Total features tested: {len(df_corr)}')
print(f'\nTop 40 features by |Spearman r| with age (healthy only):')
print('=' * 85)
for _, row in df_corr.head(40).iterrows():
    sig = '***' if row['spearman_p'] < 0.001 else '**' if row['spearman_p'] < 0.01 else '*' if row['spearman_p'] < 0.05 else ''
    print(f"{row['feature']:45s}  r={row['spearman_r']:+.3f}  pearson={row['pearson_r']:+.3f}  n={row['n']:3d}  {sig}")

In [ ]:
# Visualize top 20 correlations
top20 = df_corr.head(20).copy()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#c0392b' if r > 0 else '#2980b9' for r in top20['spearman_r']]
ax.barh(range(len(top20)), top20['spearman_r'].values, color=colors)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['feature'].values, fontsize=9)
ax.set_xlabel('Spearman r with Age (healthy population)')
ax.set_title('Top 20 Age Correlates — Healthy Population Only\n(Core Tasks: d1, d2, nb1, nb2, rest_closed)')
ax.axvline(0, color='black', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('physician_guide_figures/brain_age_nb_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: brain_age_nb_correlations.png')

In [ ]:
# Scatter plots of top 8 features vs age
top8 = df_corr.head(8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(top8.iterrows()):
    ax = axes[i]
    feat = row['feature']
    valid = df_h[[feat, 'age']].dropna()
    ax.scatter(valid['age'], valid[feat], alpha=0.4, s=15, c='#2c3e50')
    # Add trend line
    z = np.polyfit(valid['age'], valid[feat], 1)
    x_line = np.linspace(valid['age'].min(), valid['age'].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2)
    ax.set_xlabel('Age')
    ax.set_ylabel(feat.replace('_', ' '))
    ax.set_title(f'r={row["spearman_r"]:+.3f}', fontsize=10)

plt.suptitle('Top 8 Age Correlates — Healthy Population', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('physician_guide_figures/brain_age_nb_top_scatters.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Understanding the Current Model's Healthy Negative Gap

The current v2-0-1 model gives healthy population a mean gap of -3.8 years. Let's understand why.

In [ ]:
# Load current model and reproduce predictions
with open('brain_age_model_v2-0-1.json') as f:
    old_model = json.load(f)

print('Current model features:')
for i, (feat, coef) in enumerate(zip(old_model['features'], old_model['coefficients'])):
    print(f'  {i+1:2d}. {feat:45s} coef={coef:+.4f}')
print(f'\nIntercept: {old_model["intercept"]:.4f}')
print(f'Bias correction: slope={old_model["bias_correction"]["slope"]:.4f}, intercept={old_model["bias_correction"]["intercept"]:.4f}')
print(f'Error margin: {old_model["error_margin"]:.2f}')

In [ ]:
# Reproduce old model predictions on our data
def predict_old_model(df_wide_row, model_config):
    """Reproduce the old model's prediction for one user."""
    features = model_config['features']
    coefficients = model_config['coefficients']
    scaler_mean = model_config['scaler_mean']
    scaler_scale = model_config['scaler_scale']
    imputation = model_config['imputation_means']
    intercept = model_config['intercept']
    
    vec = []
    for feat_name in features:
        # Parse feature name: e.g. 'responsetime_Tasknb2' or 'responsetime_Delta_d2_d1'
        if '_Delta_' in feat_name:
            # Difference feature
            parts = feat_name.split('_Delta_')
            base = parts[0]  # e.g. 'responsetime'
            tasks = parts[1].split('_')  # e.g. ['d2', 'd1']
            col1 = f'{base}_{tasks[0]}'
            col2 = f'{base}_{tasks[1]}'
            v1 = df_wide_row.get(col1, np.nan)
            v2 = df_wide_row.get(col2, np.nan)
            if pd.notna(v1) and pd.notna(v2):
                val = v1 - v2
            else:
                val = imputation.get(feat_name, 0)
        elif '_Task' in feat_name:
            parts = feat_name.split('_Task')
            base = parts[0]
            task = parts[1]
            col = f'{base}_{task}'
            val = df_wide_row.get(col, np.nan)
            if pd.isna(val):
                val = imputation.get(feat_name, 0)
        else:
            val = imputation.get(feat_name, 0)
        vec.append(float(val))
    
    # Normalize and score
    score = intercept
    for i, v in enumerate(vec):
        normalized = (v - scaler_mean[i]) / scaler_scale[i]
        score += normalized * coefficients[i]
    
    return score

# Apply old model to all users with age
df_eval = df_wide[has_age & not_parkinsons].copy()
df_eval['old_pred_raw'] = df_eval.apply(lambda r: predict_old_model(r, old_model), axis=1)

# Apply bias correction
bc = old_model['bias_correction']
df_eval['old_bias'] = bc['intercept'] + bc['slope'] * df_eval['age']
df_eval['old_pred'] = df_eval['old_pred_raw'] - df_eval['old_bias']
df_eval['old_pred'] = df_eval['old_pred'].clip(20, 98)
df_eval['old_gap'] = df_eval['old_pred'] - df_eval['age']

# Results by group
for group_name, mask in [('Healthy', is_healthy), ('MCI', is_mci), ('MD', is_md), ('Unclassified', is_unclassified)]:
    sub = df_eval[df_eval.index.isin(df_wide[mask & has_age & not_parkinsons].index)]
    if len(sub) > 0:
        r_val, _ = pearsonr(sub['age'], sub['old_pred'])
        mae = mean_absolute_error(sub['age'], sub['old_pred'])
        print(f'{group_name:15s}  N={len(sub):4d}  r={r_val:.3f}  MAE={mae:.1f}  mean_gap={sub["old_gap"].mean():+.1f}  std_gap={sub["old_gap"].std():.1f}')

In [ ]:
# Diagnosis: Plot gap vs age for healthy — is there an age-dependent bias?
df_h_eval = df_eval[df_eval.index.isin(df_wide[healthy_mask].index)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predicted vs actual
ax = axes[0]
ax.scatter(df_h_eval['age'], df_h_eval['old_pred'], alpha=0.4, s=15)
ax.plot([20, 100], [20, 100], 'k--', alpha=0.5)
r_val, _ = pearsonr(df_h_eval['age'], df_h_eval['old_pred'])
ax.set_xlabel('Chronological Age')
ax.set_ylabel('Predicted Brain Age')
ax.set_title(f'Current Model — Healthy (r={r_val:.3f})')

# 2. Gap vs age
ax = axes[1]
ax.scatter(df_h_eval['age'], df_h_eval['old_gap'], alpha=0.4, s=15)
ax.axhline(0, color='red', linewidth=1)
z = np.polyfit(df_h_eval['age'], df_h_eval['old_gap'], 1)
x_line = np.linspace(20, 95, 100)
ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2)
ax.set_xlabel('Chronological Age')
ax.set_ylabel('Brain Age Gap (predicted - actual)')
ax.set_title(f'Gap vs Age — slope={z[0]:.3f}')

# 3. Gap distribution
ax = axes[2]
ax.hist(df_h_eval['old_gap'], bins=30, alpha=0.7, color='steelblue', edgecolor='white')
ax.axvline(df_h_eval['old_gap'].mean(), color='red', linewidth=2, label=f'Mean={df_h_eval["old_gap"].mean():.1f}')
ax.set_xlabel('Brain Age Gap')
ax.set_ylabel('Count')
ax.set_title('Gap Distribution — Healthy')
ax.legend()

plt.suptitle('Current Model v2-0-1 Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nHealthy gap: mean={df_h_eval["old_gap"].mean():.2f}, median={df_h_eval["old_gap"].median():.2f}')
print(f'Gap vs age correlation: r={spearmanr(df_h_eval["age"], df_h_eval["old_gap"])[0]:.3f}')

## 4. Build New Models

Strategy:
1. **Train only on healthy + unclassified** (exclude MCI, MD, Parkinson's)
2. Use proper CV (leave-one-out or 10-fold) to get unbiased predictions
3. Try multiple model types: Ridge, Lasso, ElasticNet, GBR
4. Select top features from the correlation analysis
5. Apply bias correction so healthy mean gap ≈ 0

In [ ]:
# Prepare training data — healthy + unclassified, with age, no Parkinson's
df_train = df_wide[train_mask].copy()
y_train = df_train['age'].values

# Get candidate features — those with |r| > 0.15 in healthy population
strong_feats = df_corr[df_corr['abs_r'] >= 0.15]['feature'].tolist()
# Also filter to features with enough coverage (>80% non-null in training set)
good_coverage = [f for f in strong_feats if df_train[f].notna().mean() > 0.80]

print(f'Features with |r|>0.15 and >80% coverage: {len(good_coverage)}')
for f in good_coverage:
    r_val = df_corr[df_corr['feature'] == f]['spearman_r'].values[0]
    cov = df_train[f].notna().mean()
    print(f'  {f:45s}  r={r_val:+.3f}  coverage={cov:.1%}')

In [ ]:
def evaluate_model(model, X, y, cv, model_name='Model'):
    """Evaluate a model with cross-validation and return predictions + stats."""
    y_pred = cross_val_predict(model, X, y, cv=cv)
    r_pearson, _ = pearsonr(y, y_pred)
    r_spearman, _ = spearmanr(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    gap = y_pred - y
    
    print(f'{model_name:30s}  r={r_pearson:.3f}  rho={r_spearman:.3f}  MAE={mae:.1f}  gap_mean={gap.mean():+.2f}  gap_std={gap.std():.1f}')
    return y_pred, r_pearson, mae

# Prepare feature matrices
# Fill NaN with column median for training
X_train_raw = df_train[good_coverage].copy()
for col in good_coverage:
    X_train_raw[col] = X_train_raw[col].fillna(X_train_raw[col].median())

X_train_arr = X_train_raw.values

cv10 = KFold(n_splits=10, shuffle=True, random_state=42)

print('=== All candidate features ===' )
print(f'N={len(y_train)}, Features={X_train_arr.shape[1]}\n')

results = {}
for name, model in [
    ('Ridge(alpha=1)', Ridge(alpha=1)),
    ('Ridge(alpha=10)', Ridge(alpha=10)),
    ('Ridge(alpha=100)', Ridge(alpha=100)),
    ('Lasso(alpha=0.1)', Lasso(alpha=0.1, max_iter=10000)),
    ('Lasso(alpha=1)', Lasso(alpha=1, max_iter=10000)),
    ('ElasticNet(0.5)', ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=10000)),
    ('ElasticNet(1.0)', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)),
    ('GBR(100,3)', GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)),
    ('GBR(200,2)', GradientBoostingRegressor(n_estimators=200, max_depth=2, random_state=42)),
]:
    pred, r, mae = evaluate_model(model, X_train_arr, y_train, cv10, name)
    results[name] = (pred, r, mae)

In [ ]:
# Now try with feature selection — top N features only
print('\n=== Feature selection experiments ===')
for n_feats in [5, 8, 10, 15, 20, 25]:
    top_n = good_coverage[:n_feats]
    X_n = df_train[top_n].copy()
    for col in top_n:
        X_n[col] = X_n[col].fillna(X_n[col].median())
    X_n_arr = X_n.values
    
    for alpha in [1, 10]:
        model = Ridge(alpha=alpha)
        pred, r, mae = evaluate_model(model, X_n_arr, y_train, cv10, f'Ridge(a={alpha}) top-{n_feats}')
    
    # Also try GBR
    model = GradientBoostingRegressor(n_estimators=100, max_depth=2, random_state=42)
    pred, r, mae = evaluate_model(model, X_n_arr, y_train, cv10, f'GBR(100,2) top-{n_feats}')
    print()

## 5. Evaluate on Healthy Population Specifically

The target is r > 0.7 on the healthy population.

In [ ]:
# For healthy-only evaluation, we need CV predictions for the healthy subset
# Best approach: train on full training set (healthy+unclassified), get CV predictions, 
# then evaluate only the healthy subset

# Let's identify which rows in df_train are healthy vs unclassified
is_healthy_in_train = df_train['mmse_group'] == 'healthy'
healthy_indices = np.where(is_healthy_in_train.values)[0]

print(f'Healthy in training set: {len(healthy_indices)}')
print(f'Unclassified in training set: {(~is_healthy_in_train).sum()}')

# Try best configurations and evaluate on healthy subset
print('\n=== Healthy-only evaluation (CV predictions from full training set) ===\n')

best_r = 0
best_config = None

for n_feats in [5, 8, 10, 15, 20, 25, len(good_coverage)]:
    top_n = good_coverage[:min(n_feats, len(good_coverage))]
    X_n = df_train[top_n].copy()
    for col in top_n:
        X_n[col] = X_n[col].fillna(X_n[col].median())
    X_n_arr = X_n.values
    
    for name, model in [
        (f'Ridge(1) top-{n_feats}', Ridge(alpha=1)),
        (f'Ridge(10) top-{n_feats}', Ridge(alpha=10)),
        (f'GBR(100,2) top-{n_feats}', GradientBoostingRegressor(n_estimators=100, max_depth=2, random_state=42)),
    ]:
        y_pred_all = cross_val_predict(model, X_n_arr, y_train, cv=cv10)
        
        # Healthy subset
        y_h = y_train[healthy_indices]
        pred_h = y_pred_all[healthy_indices]
        r_h, _ = pearsonr(y_h, pred_h)
        mae_h = mean_absolute_error(y_h, pred_h)
        gap_h = (pred_h - y_h).mean()
        
        # All training
        r_all, _ = pearsonr(y_train, y_pred_all)
        mae_all = mean_absolute_error(y_train, y_pred_all)
        
        marker = ' <<<' if r_h > best_r else ''
        print(f'{name:30s}  ALL: r={r_all:.3f} MAE={mae_all:.1f}  |  HEALTHY: r={r_h:.3f} MAE={mae_h:.1f} gap={gap_h:+.1f}{marker}')
        
        if r_h > best_r:
            best_r = r_h
            best_config = (name, n_feats, model, top_n)
    print()

print(f'\nBest healthy r: {best_r:.3f} with {best_config[0]}')

In [ ]:
# Try training ONLY on healthy population (more homogeneous, 
# might give better r within that group)
print('=== Training on HEALTHY ONLY ===\n')

df_h_train = df_wide[healthy_mask].copy()
y_h_train = df_h_train['age'].values

best_r_ho = 0
best_config_ho = None

for n_feats in [5, 8, 10, 15, 20, 25, len(good_coverage)]:
    top_n = good_coverage[:min(n_feats, len(good_coverage))]
    X_n = df_h_train[top_n].copy()
    for col in top_n:
        X_n[col] = X_n[col].fillna(X_n[col].median())
    X_n_arr = X_n.values
    
    for aname, model in [
        (f'Ridge(1) top-{n_feats}', Ridge(alpha=1)),
        (f'Ridge(10) top-{n_feats}', Ridge(alpha=10)),
        (f'Ridge(100) top-{n_feats}', Ridge(alpha=100)),
        (f'Lasso(0.5) top-{n_feats}', Lasso(alpha=0.5, max_iter=10000)),
        (f'ElasticNet top-{n_feats}', ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=10000)),
        (f'GBR(100,2) top-{n_feats}', GradientBoostingRegressor(n_estimators=100, max_depth=2, random_state=42)),
        (f'GBR(200,3) top-{n_feats}', GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42)),
    ]:
        y_pred = cross_val_predict(model, X_n_arr, y_h_train, cv=cv10)
        r_val, _ = pearsonr(y_h_train, y_pred)
        mae = mean_absolute_error(y_h_train, y_pred)
        gap = (y_pred - y_h_train).mean()
        
        marker = ' <<<' if r_val > best_r_ho else ''
        print(f'{aname:30s}  r={r_val:.3f}  MAE={mae:.1f}  gap={gap:+.2f}{marker}')
        
        if r_val > best_r_ho:
            best_r_ho = r_val
            best_config_ho = (aname, n_feats, model, top_n)
    print()

print(f'\nBest healthy-only r: {best_r_ho:.3f} with {best_config_ho[0]}')

## 6. Final Model Selection & Bias Correction

Based on the experiments above, select the best model configuration.
Then apply proper bias correction so that:
- Healthy mean gap ≈ 0
- Gap increases for MCI and MD

In [ ]:
# =====================================================
# SELECT BEST MODEL CONFIGURATION HERE
# After running above cells, pick the best approach.
# We'll try two strategies and compare:
#   A) Train on healthy+unclassified, evaluate on healthy
#   B) Train on healthy only, evaluate on healthy
# =====================================================

# Strategy A: Train on all healthy+unclassified
# Strategy B: Train on healthy only
# Let's finalize both and compare

# For production, we want:
# 1. Best r on healthy
# 2. Mean gap ≈ 0 for healthy (via bias correction)
# 3. Positive gap for MCI/MD
# 4. Lean model (few features, linear preferred for OSP real-time)

# Let's pick the top feature set and Ridge (linear, fast, reproducible)
# and compare with GBR (slightly better but harder to deploy)

# APPROACH: Use top N features, Ridge regression, trained on healthy+unclassified
# with bias correction fit on healthy-only CV residuals

# Try the definitive configuration
print('=== DEFINITIVE MODEL COMPARISON ===\n')

for n_feats in [10, 15, 20]:
    feats = good_coverage[:n_feats]
    
    for train_name, train_df, train_y in [
        ('healthy+unclassified', df_train, y_train),
        ('healthy-only', df_h_train, y_h_train),
    ]:
        X = train_df[feats].copy()
        for col in feats:
            X[col] = X[col].fillna(X[col].median())
        X_arr = X.values
        
        for alpha in [1, 10]:
            model = Ridge(alpha=alpha)
            y_pred = cross_val_predict(model, X_arr, train_y, cv=cv10)
            
            # If trained on healthy+unclassified, get healthy subset
            if train_name == 'healthy+unclassified':
                h_idx = np.where(is_healthy_in_train.values)[0]
                r_h, _ = pearsonr(train_y[h_idx], y_pred[h_idx])
                mae_h = mean_absolute_error(train_y[h_idx], y_pred[h_idx])
                gap_h = (y_pred[h_idx] - train_y[h_idx]).mean()
            else:
                r_h, _ = pearsonr(train_y, y_pred)
                mae_h = mean_absolute_error(train_y, y_pred)
                gap_h = (y_pred - train_y).mean()
            
            print(f'Ridge(a={alpha:3d}) {n_feats:2d}F {train_name:25s}  healthy: r={r_h:.3f} MAE={mae_h:.1f} gap={gap_h:+.1f}')
    print()

In [ ]:
# =====================================================
# FINAL MODEL: Fill in based on best result above
# =====================================================

# Configuration — EDIT THESE based on results above
FINAL_N_FEATS = 15  # adjust based on best result
FINAL_ALPHA = 10    # adjust based on best result  
TRAIN_ON = 'healthy+unclassified'  # or 'healthy-only'

final_feats = good_coverage[:FINAL_N_FEATS]

if TRAIN_ON == 'healthy+unclassified':
    df_final_train = df_train.copy()
    y_final = df_final_train['age'].values
else:
    df_final_train = df_h_train.copy()
    y_final = df_final_train['age'].values

X_final = df_final_train[final_feats].copy()
imputation_means = {}
for col in final_feats:
    med = X_final[col].median()
    imputation_means[col] = float(med)
    X_final[col] = X_final[col].fillna(med)

X_final_arr = X_final.values

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final_arr)

# CV predictions for evaluation
final_model = Ridge(alpha=FINAL_ALPHA)
y_pred_cv = cross_val_predict(final_model, X_scaled, y_final, cv=cv10)

# Fit final model on all data
final_model.fit(X_scaled, y_final)

print(f'Final model: Ridge(alpha={FINAL_ALPHA}), {FINAL_N_FEATS} features, trained on {TRAIN_ON}')
print(f'Intercept: {final_model.intercept_:.4f}')
print(f'\nFeature coefficients:')
for feat, coef in sorted(zip(final_feats, final_model.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {feat:45s}  coef={coef:+.4f}')

In [ ]:
# Bias correction: fit linear model on CV residuals so healthy gap → 0
# raw_gap = y_pred_cv - y_final
# We want: corrected = raw_pred - (a + b * age) such that E[corrected - age | healthy] ≈ 0

# Fit bias on the full training set CV predictions
from sklearn.linear_model import LinearRegression

bias_model = LinearRegression()
residuals = y_pred_cv - y_final  # raw prediction errors
bias_model.fit(y_final.reshape(-1, 1), residuals)

bias_slope = bias_model.coef_[0]
bias_intercept = bias_model.intercept_

print(f'Bias correction: gap = {bias_intercept:.4f} + {bias_slope:.4f} * age')
print(f'This means: corrected_pred = raw_pred - ({bias_intercept:.4f} + {bias_slope:.4f} * age)')

# Apply bias correction to CV predictions
bias = bias_intercept + bias_slope * y_final
y_corrected = y_pred_cv - bias

# Evaluate corrected predictions
r_all, _ = pearsonr(y_final, y_corrected)
mae_all = mean_absolute_error(y_final, y_corrected)
gap_all = (y_corrected - y_final).mean()
print(f'\nAll training: r={r_all:.3f}, MAE={mae_all:.1f}, mean gap={gap_all:+.2f}')

# Healthy subset
if TRAIN_ON == 'healthy+unclassified':
    h_idx = np.where(is_healthy_in_train.values)[0]
    r_h, _ = pearsonr(y_final[h_idx], y_corrected[h_idx])
    mae_h = mean_absolute_error(y_final[h_idx], y_corrected[h_idx])
    gap_h = (y_corrected[h_idx] - y_final[h_idx]).mean()
    print(f'Healthy:      r={r_h:.3f}, MAE={mae_h:.1f}, mean gap={gap_h:+.2f}')
else:
    r_h = r_all
    mae_h = mae_all
    gap_h = gap_all
    print(f'(Trained on healthy only — same as above)')

In [ ]:
# Now predict on MCI and MD to see the gap gradient
# We need to apply the final trained model (not CV) to out-of-sample groups

def predict_new_model(df_subset, feats, scaler, model, bias_slope, bias_intercept, imputation_means):
    """Apply the new model to a subset of users."""
    X = df_subset[feats].copy()
    for col in feats:
        X[col] = X[col].fillna(imputation_means[col])
    X_scaled = scaler.transform(X.values)
    raw_pred = model.predict(X_scaled)
    ages = df_subset['age'].values
    bias = bias_intercept + bias_slope * ages
    corrected = raw_pred - bias
    corrected = np.clip(corrected, 20, 98)
    return corrected

# Evaluate on each group
print('=== NEW MODEL — Group Results ===\n')
print(f'Model: Ridge(alpha={FINAL_ALPHA}), {FINAL_N_FEATS} features\n')

for gname, gmask in [
    ('Healthy', healthy_mask),
    ('MCI', has_age & is_mci & not_parkinsons),
    ('Dementia', has_age & is_md & not_parkinsons),
    ('Unclassified', has_age & is_unclassified & not_parkinsons),
]:
    sub = df_wide[gmask].copy()
    if len(sub) == 0:
        continue
    pred = predict_new_model(sub, final_feats, scaler, final_model, bias_slope, bias_intercept, imputation_means)
    ages = sub['age'].values
    gap = pred - ages
    r_val, _ = pearsonr(ages, pred)
    mae = mean_absolute_error(ages, pred)
    print(f'{gname:15s}  N={len(sub):4d}  r={r_val:.3f}  MAE={mae:.1f}  gap={gap.mean():+.1f} +/- {gap.std():.1f}  median_gap={np.median(gap):+.1f}')

# Age-matched comparison (60-85)
print('\n=== Age-matched (60-85) ===\n')
for gname, gmask in [
    ('Healthy', healthy_mask),
    ('MCI', has_age & is_mci & not_parkinsons),
    ('Dementia', has_age & is_md & not_parkinsons),
]:
    sub = df_wide[gmask & (df_wide['age'] >= 60) & (df_wide['age'] <= 85)].copy()
    if len(sub) == 0:
        continue
    pred = predict_new_model(sub, final_feats, scaler, final_model, bias_slope, bias_intercept, imputation_means)
    ages = sub['age'].values
    gap = pred - ages
    print(f'{gname:15s}  N={len(sub):4d}  mean_gap={gap.mean():+.1f} +/- {gap.std():.1f}')

In [ ]:
# Visualization of the final model
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Predicted vs Actual — Healthy (CV predictions)
ax = axes[0, 0]
if TRAIN_ON == 'healthy+unclassified':
    h_ages = y_final[healthy_indices]
    h_pred = y_corrected[healthy_indices]
else:
    h_ages = y_final
    h_pred = y_corrected

ax.scatter(h_ages, h_pred, alpha=0.4, s=20, c='#27ae60', label='Healthy')
ax.plot([20, 100], [20, 100], 'k--', alpha=0.5, label='Perfect prediction')
z = np.polyfit(h_ages, h_pred, 1)
x_line = np.linspace(20, 95, 100)
ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2, label=f'Fit (r={pearsonr(h_ages, h_pred)[0]:.3f})')
ax.set_xlabel('Chronological Age', fontsize=12)
ax.set_ylabel('Predicted Brain Age', fontsize=12)
ax.set_title('New Model — Healthy Population (CV)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# 2. Gap vs Age — Healthy
ax = axes[0, 1]
gap_h = h_pred - h_ages
ax.scatter(h_ages, gap_h, alpha=0.4, s=20, c='#27ae60')
ax.axhline(0, color='red', linewidth=1)
z2 = np.polyfit(h_ages, gap_h, 1)
ax.plot(x_line, np.polyval(z2, x_line), 'r-', linewidth=2, label=f'slope={z2[0]:.3f}')
ax.set_xlabel('Chronological Age', fontsize=12)
ax.set_ylabel('Brain Age Gap', fontsize=12)
ax.set_title(f'Gap vs Age — Healthy (mean gap={gap_h.mean():+.1f})', fontsize=13, fontweight='bold')
ax.legend()

# 3. Predicted vs Actual — All groups
ax = axes[1, 0]
for gname, gmask, color in [
    ('Healthy', healthy_mask, '#27ae60'),
    ('Unclassified', has_age & is_unclassified & not_parkinsons, '#3498db'),
    ('MCI', has_age & is_mci & not_parkinsons, '#e67e22'),
    ('Dementia', has_age & is_md & not_parkinsons, '#e74c3c'),
]:
    sub = df_wide[gmask].copy()
    if len(sub) == 0:
        continue
    pred = predict_new_model(sub, final_feats, scaler, final_model, bias_slope, bias_intercept, imputation_means)
    ax.scatter(sub['age'].values, pred, alpha=0.3, s=15, c=color, label=gname)

ax.plot([20, 100], [20, 100], 'k--', alpha=0.5)
ax.set_xlabel('Chronological Age', fontsize=12)
ax.set_ylabel('Predicted Brain Age', fontsize=12)
ax.set_title('All Clinical Groups', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# 4. Gap by clinical group (age-matched 60-85)
ax = axes[1, 1]
gap_data = []
gap_labels = []
gap_colors = []
for gname, gmask, color in [
    ('Healthy', healthy_mask, '#27ae60'),
    ('MCI', has_age & is_mci & not_parkinsons, '#e67e22'),
    ('Dementia', has_age & is_md & not_parkinsons, '#e74c3c'),
]:
    sub = df_wide[gmask & (df_wide['age'] >= 60) & (df_wide['age'] <= 85)].copy()
    if len(sub) == 0:
        continue
    pred = predict_new_model(sub, final_feats, scaler, final_model, bias_slope, bias_intercept, imputation_means)
    gap = pred - sub['age'].values
    gap_data.append(gap)
    gap_labels.append(f'{gname}\n(N={len(sub)})')
    gap_colors.append(color)

bp = ax.boxplot(gap_data, labels=gap_labels, patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], gap_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(0, color='red', linewidth=1, linestyle='--')
# Add mean markers
for i, d in enumerate(gap_data):
    ax.plot(i+1, d.mean(), 'D', color='black', markersize=8, zorder=5)
    ax.annotate(f'{d.mean():+.1f}', (i+1, d.mean()), textcoords='offset points', 
                xytext=(15, 0), fontsize=11, fontweight='bold')
ax.set_ylabel('Brain Age Gap (years)', fontsize=12)
ax.set_title('Brain Age Gap by Group (age-matched 60-85)', fontsize=13, fontweight='bold')

plt.suptitle(f'New Brain Age Model v3.0 — Ridge({FINAL_ALPHA}), {FINAL_N_FEATS} features', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('physician_guide_figures/brain_age_new_model_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Model for OSP

Save the final model in the same JSON format as v2-0-1 so the OSP code can load it.

In [ ]:
# Convert feature names to OSP format
# Our columns: e.g. 'A0_rest_closed', 'responsetime_nb2', 'Alpha_diff_nb2_rest'
# OSP format: e.g. 'A0_Taskrest_closed', 'responsetime_Tasknb2', 'Alpha_Delta_nb2_rest'

def to_osp_feature_name(col_name):
    """Convert our feature column name to OSP model format."""
    if '_diff_' in col_name:
        # Difference feature: e.g. 'A0_diff_nb2_nb1' -> 'A0_Delta_nb2_nb1'
        parts = col_name.split('_diff_')
        return f'{parts[0]}_Delta_{parts[1]}'
    elif '_mean_all' in col_name:
        # Mean across all tasks: e.g. 'A0_mean_all' -> 'A0_MeanAll'
        base = col_name.replace('_mean_all', '')
        return f'{base}_MeanAll'
    else:
        # Task-specific: e.g. 'A0_rest_closed' -> 'A0_Taskrest_closed'
        # Find where the task name starts
        for task in ['rest_closed', 'nb1', 'nb2', 'd1', 'd2']:
            if col_name.endswith(f'_{task}'):
                base = col_name[:-(len(task)+1)]
                return f'{base}_Task{task}'
        return col_name  # fallback

osp_feature_names = [to_osp_feature_name(f) for f in final_feats]
print('Feature name mapping:')
for orig, osp in zip(final_feats, osp_feature_names):
    print(f'  {orig:45s} -> {osp}')

In [ ]:
# Build the model config JSON
import datetime

# Compute error margin from CV residuals
if TRAIN_ON == 'healthy+unclassified':
    cv_residuals = y_corrected[healthy_indices] - y_final[healthy_indices]
else:
    cv_residuals = y_corrected - y_final
error_margin = float(np.percentile(np.abs(cv_residuals), 75))  # 75th percentile of |error|

# Build imputation means in OSP format
osp_imputation = {osp_name: imputation_means[orig_name] 
                  for orig_name, osp_name in zip(final_feats, osp_feature_names)}

model_config = {
    'version': '3.0.0',
    'created': datetime.datetime.now().isoformat(),
    'description': f'Brain Age model v3.0 — Ridge(alpha={FINAL_ALPHA}), {FINAL_N_FEATS} features, trained on {TRAIN_ON}',
    'features': osp_feature_names,
    'coefficients': [float(c) for c in final_model.coef_],
    'intercept': float(final_model.intercept_),
    'scaler_mean': [float(m) for m in scaler.mean_],
    'scaler_scale': [float(s) for s in scaler.scale_],
    'bias_correction': {
        'slope': float(bias_slope),
        'intercept': float(bias_intercept)
    },
    'error_margin': float(error_margin),
    'imputation_means': osp_imputation,
    'required_tasks': ['d1', 'd2', 'nb1', 'nb2', 'rest_closed'],
    'training_info': {
        'n_train': int(len(y_final)),
        'n_healthy': int(healthy_mask.sum()),
        'train_population': TRAIN_ON,
        'cv_r_healthy': float(r_h),
        'cv_mae_healthy': float(mae_h),
        'alpha': FINAL_ALPHA
    }
}

# Save
out_path = 'brain_age_model_v3-0-0.json'
with open(out_path, 'w') as f:
    json.dump(model_config, f, indent=2)

print(f'Saved model config: {out_path}')
print(f'\nModel summary:')
print(f'  Features: {FINAL_N_FEATS}')
print(f'  Healthy r: {r_h:.3f}')
print(f'  Healthy MAE: {mae_h:.1f}')
print(f'  Error margin: {error_margin:.1f} years')
print(f'  Bias correction: {bias_intercept:.4f} + {bias_slope:.4f} * age')

## 8. Summary & Conclusions

Fill in after running all cells above.

In [ ]:
# Final summary
print('=' * 70)
print('BRAIN AGE MODEL v3.0 — FINAL SUMMARY')
print('=' * 70)
print(f'\nModel type: Ridge regression (alpha={FINAL_ALPHA})')
print(f'Features: {FINAL_N_FEATS} (from core tasks: d1, d2, nb1, nb2, rest_closed)')
print(f'Training population: {TRAIN_ON}')
print(f'\nPerformance (10-fold CV):')
print(f'  Healthy population:  r = {r_h:.3f},  MAE = {mae_h:.1f} years')
print(f'  Healthy mean gap:    {gap_h:+.2f} years (target: ~0)')
print(f'  Error margin (75p):  +/- {error_margin:.1f} years')
print(f'\nBias correction: pred_corrected = pred_raw - ({bias_intercept:.4f} + {bias_slope:.4f} * real_age)')
print(f'\nTop features by |coefficient|:')
sorted_feats = sorted(zip(final_feats, osp_feature_names, final_model.coef_), 
                       key=lambda x: abs(x[2]), reverse=True)
for orig, osp, coef in sorted_feats[:10]:
    print(f'  {osp:45s}  coef={coef:+.4f}')

print(f'\n{"=" * 70}')
print('NEXT STEPS:')
print('  1. Copy brain_age_model_v3-0-0.json to OSP directory')
print('  2. Update config.py BRAIN_AGE_MODEL reference')
print('  3. Update brain_age.py if feature name format changed')
print('  4. Update physician guide with new figures and stats')
print(f'{"=" * 70}')